In [1]:
!pip install selenium pandas webdriver-manager BeautifulSoup4

In [3]:
chrome.exe --remote-debugging-port=9222 --user-data-dir="C:\chrometemp"

SyntaxError: cannot assign to expression (3431857801.py, line 1)

In [51]:
import time
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# -------------------------------------------------------------------------
# [설정 항목] 진입 URL 및 키워드 세팅 (날짜 제한 없음)
# -------------------------------------------------------------------------
TARGET_URL = "https://everytime.kr/"

SEARCH_KEYWORDS = [
    "기숙사 짐", 
    "방학 짐", 
    "본가 택배", 
    "본가 짐", 
    "짐보관", 
    "짐 택배"
]

# -------------------------------------------------------------------------
# 1. 디버거 크롬 브라우저 연동 및 메인 홈 진입
# -------------------------------------------------------------------------
chrome_options = Options()
chrome_options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
wait = WebDriverWait(driver, 12)

print("🔗 [성공] 제어 가능한 디버그 크롬 브라우저 동기화 완료!")
print(f"🌐 입력하신 베이스 URL 주소로 이동합니다 -> {TARGET_URL}")
driver.get(TARGET_URL)
time.sleep(3.5)

scraped_data = []

# -------------------------------------------------------------------------
# 2. 키워드 순차 검색 및 페이징 탐색 루프 (연도/날짜 무제한 고속 수집)
# -------------------------------------------------------------------------
for keyword in SEARCH_KEYWORDS:
    print(f"\n🔎 [{keyword}] 검색 및 페이징 한정 해제 전수 수집 시작...")
    
    try:
        search_input = wait.until(EC.visibility_of_element_located(
            (By.CSS_SELECTOR, "#container > div.rightside > form > input")
        ))
        
        search_input.click()
        time.sleep(0.3)
        search_input.send_keys(Keys.CONTROL + "a")
        search_input.send_keys(Keys.DELETE)
        time.sleep(0.3)
        
        search_input.send_keys(keyword)
        time.sleep(0.4)
        search_input.send_keys(Keys.ENTER)
        
        print(f"   -> 엔터 입력 완료. 결과 페이지 전환 및 데이터 로드 대기 중...")
        time.sleep(4.0)
        
        page_depth = 1
        
        while True:
            print(f"   -> 🛑 [{page_depth}페이지] 내부 박스(#container) 끝까지 스크롤 다운 중...")
            
            # [A] 내부 스크롤 끝까지 내리기 (7개 한정 해제)
            last_height = driver.execute_script("return document.querySelector('#container') ? document.querySelector('#container').scrollHeight : 0")
            while True:
                driver.execute_script("""
                    var container = document.querySelector('#container');
                    if(container) { container.scrollTo(0, container.scrollHeight); }
                """)
                time.sleep(1.4)
                new_height = driver.execute_script("return document.querySelector('#container') ? document.querySelector('#container').scrollHeight : 0")
                if new_height == last_height:
                    break
                last_height = new_height
            
            # [B] BeautifulSoup 파싱 시작
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            articles = soup.select('article')
            if not articles:
                articles = soup.select('a.article')
                
            match_count = 0
            for article in articles:
                try:
                    link_tag = article if article.name == 'a' else article.select_one('a')
                    if not link_tag: continue
                        
                    if 'data-id' in link_tag.attrs:
                        content_id = int(link_tag['data-id'])
                    elif 'href' in link_tag.attrs:
                        content_id = int(link_tag['href'].split('/')[-1])
                    else: continue
                    
                    title_tag = article.select_one('h2')
                    title = title_tag.get_text(strip=True) if title_tag else ""
                    
                    text_tag = article.select_one('p')
                    content_text = text_tag.get_text(strip=True) if text_tag else ""
                    
                    # 날짜 텍스트 추출 (필터링 없이 텍스트 그대로 원본 보존)
                    date_tag = article.select_one('time')
                    date_str = date_tag.get_text(strip=True) if date_tag else ""
                    
                    comment_tag = article.select_one('.comment')
                    if comment_tag:
                        comment_text_raw = comment_tag.get_text(strip=True)
                        comment_count = int(comment_text_raw) if comment_text_raw.isdigit() else 0
                    else:
                        comment_count = 0
                    
                    # 중복 저장 방지 (여러 페이지 처리 시 인접 페이지 중복 유입 방지)
                    if any(d['content_id'] == content_id for d in scraped_data):
                        continue
                    
                    scraped_data.append({
                        "content_id": content_id,
                        "platform": "everytime",
                        "keyword": keyword,
                        "title": title,
                        "content_text": content_text,
                        "comment_count": comment_count,
                        "posted_at": date_str  # 화면에 표시된 날짜 원본 텍스트 그대로 매핑
                    })
                    match_count += 1
                    
                except Exception:
                    continue
            
            print(f"   -> [{page_depth}페이지] 수집 완료: {match_count}개 추가 저장됨")
            
            # [C] [다음] 페이지 이동 처리
            try:
                pagination_buttons = driver.find_elements(By.CSS_SELECTOR, "#container > div.wrap.articles > div.pagination > a")
                
                next_btn = None
                for btn in pagination_buttons:
                    if btn.text.strip() == "다음":
                        next_btn = btn
                        break
                
                if next_btn and next_btn.is_displayed():
                    print("   -> ➡️ [다음] 버튼을 클릭하여 과거 게시글 페이지로 이동합니다.")
                    driver.execute_script("arguments[0].click();", next_btn)
                    page_depth += 1
                    time.sleep(4.0)
                else:
                    print(f"   -> ✨ [끝] 더 이상 클릭할 '다음' 버튼이 없습니다. [{keyword}] 최종 종료.")
                    break
            except Exception:
                print(f"   -> ✨ [끝] 마지막 페이지에 도달했습니다. [{keyword}] 최종 종료.")
                break
        
        driver.get(TARGET_URL)
        time.sleep(3.0)
                
    except Exception as e:
        print(f"❌ [{keyword}] 요소 제어 중 에러 발생: {e}")
        driver.get(TARGET_URL)
        time.sleep(4.0)
        continue

# -------------------------------------------------------------------------
# 3. 종합 데이터 CSV 저장 단계
# -------------------------------------------------------------------------
df = pd.DataFrame(scraped_data)

if not df.empty:
    output_filename = f"everytime_unlimited_posts_{datetime.now().strftime('%Y%m%d')}.csv"
    df.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"\n🎉 [최종 대성공] 연도 무제한 전수 수집 완수! 총 {len(df)}건 데이터가 '{output_filename}' 파일로 저장되었습니다.")
else:
    print("\n⚠️ 탐색은 종료되었으나 데이터가 수집되지 않았습니다. 브라우저 로그인 세션을 확인해 주세요.")

🔗 [성공] 제어 가능한 디버그 크롬 브라우저 동기화 완료!
🌐 입력하신 베이스 URL 주소로 이동합니다 -> https://everytime.kr/

🔎 [기숙사 짐] 검색 및 페이징 한정 해제 전수 수집 시작...
   -> 엔터 입력 완료. 결과 페이지 전환 및 데이터 로드 대기 중...
   -> 🛑 [1페이지] 내부 박스(#container) 끝까지 스크롤 다운 중...
   -> [1페이지] 수집 완료: 20개 추가 저장됨
   -> ➡️ [다음] 버튼을 클릭하여 과거 게시글 페이지로 이동합니다.
   -> 🛑 [2페이지] 내부 박스(#container) 끝까지 스크롤 다운 중...
   -> [2페이지] 수집 완료: 20개 추가 저장됨
   -> ➡️ [다음] 버튼을 클릭하여 과거 게시글 페이지로 이동합니다.
   -> 🛑 [3페이지] 내부 박스(#container) 끝까지 스크롤 다운 중...
   -> [3페이지] 수집 완료: 20개 추가 저장됨
   -> ➡️ [다음] 버튼을 클릭하여 과거 게시글 페이지로 이동합니다.
   -> 🛑 [4페이지] 내부 박스(#container) 끝까지 스크롤 다운 중...
   -> [4페이지] 수집 완료: 20개 추가 저장됨
   -> ➡️ [다음] 버튼을 클릭하여 과거 게시글 페이지로 이동합니다.
   -> 🛑 [5페이지] 내부 박스(#container) 끝까지 스크롤 다운 중...
   -> [5페이지] 수집 완료: 20개 추가 저장됨
   -> ➡️ [다음] 버튼을 클릭하여 과거 게시글 페이지로 이동합니다.
   -> 🛑 [6페이지] 내부 박스(#container) 끝까지 스크롤 다운 중...
   -> [6페이지] 수집 완료: 20개 추가 저장됨
   -> ➡️ [다음] 버튼을 클릭하여 과거 게시글 페이지로 이동합니다.
   -> 🛑 [7페이지] 내부 박스(#container) 끝까지 스크롤 다운 중...
   -> [7페이지] 수집 완료: 13개 추가 저장됨
   -> ➡️ [다음] 버튼을 클릭하여 

In [46]:
#최종KW2) 단기임대, 방양도, 원룸+양도, 오피스텔+양도, 자취방+양도 크롤링

import time
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# -------------------------------------------------------------------------
# [설정 항목] 검색어 목록 세팅
# -------------------------------------------------------------------------
SEARCH_KEYWORDS = [
    "단기 임대",
    "방 양도",
    "원룸 양도",
    "오피스텔 양도",
    "자취방 양도"
]

# -------------------------------------------------------------------------
# 1. 디버거 크롬 브라우저 다이렉트 연동
# -------------------------------------------------------------------------
chrome_options = Options()
chrome_options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")

try:
    driver = webdriver.Chrome(options=chrome_options)
    wait = WebDriverWait(driver, 8)
    print(f"🔗 [연동 성공] 디버그 크롬 실시간 제어권 획득 완료!")
except Exception as connect_error:
    print(f"❌ 크롬 브라우저 통신 실패: {connect_error}")
    raise

scraped_data = []

print("🌐 안전한 메인 홈 화면으로 진입합니다.")
driver.get("https://everytime.kr/")
time.sleep(3.5)

# -------------------------------------------------------------------------
# 2. 키워드별 [내부 스크롤 + 다음 버튼 연쇄 클릭] 전수 조사
# -------------------------------------------------------------------------
for keyword in SEARCH_KEYWORDS:
    print(f"\n🔎 [{keyword}] 검색 및 페이징 한정 해제 전수 수집 시작...")
    
    try:
        # 우측 핀포인트 검색바 대기 및 타겟팅
        search_input = wait.until(EC.element_to_be_clickable(
            (By.CSS_SELECTOR, "#container > div.rightside > form > input")
        ))
        
        search_input.click()
        time.sleep(0.3)
        search_input.send_keys(Keys.CONTROL + "a")
        search_input.send_keys(Keys.DELETE)
        time.sleep(0.3)
        
        search_input.send_keys(keyword)
        time.sleep(0.4)
        search_input.send_keys(Keys.ENTER)
        
        print("   -> 엔터 입력 완료. 검색 결과 첫 페이지 로딩 대기...")
        time.sleep(4.0)
        
        page_depth = 1  # 현재 몇 번째 페이지(다음 버튼 클릭 횟수)인지 추적
        
        # [핵심] 다음 버튼이 없을 때까지 계속 반복하는 거대한 루프
        while True:
            print(f"   -> 🛑 [{page_depth}층] 내부 박스 끝까지 스크롤 다운 중...")
            
            # 1) 해당 페이지 안에서 마우스 스크롤을 끝까지 내립니다.
            last_height = driver.execute_script("return document.querySelector('#container') ? document.querySelector('#container').scrollHeight : 0")
            while True:
                driver.execute_script("""
                    var container = document.querySelector('#container');
                    if(container) { container.scrollTo(0, container.scrollHeight); }
                """)
                time.sleep(1.4)
                new_height = driver.execute_script("return document.querySelector('#container') ? document.querySelector('#container').scrollHeight : 0")
                if new_height == last_height:
                    break
                last_height = new_height
            
            # 2) 현재 스크롤이 다 내려간 상태의 HTML 데이터를 수집 대상에 임시 파싱 추가
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            articles = soup.select('article')
            if not articles:
                articles = soup.select('a.article')
                
            match_count = 0
            for article in articles:
                try:
                    link_tag = article if article.name == 'a' else article.select_one('a')
                    if not link_tag: continue
                    
                    if 'data-id' in link_tag.attrs:
                        content_id = int(link_tag['data-id'])
                    elif 'href' in link_tag.attrs:
                        content_id = int(link_tag['href'].split('/')[-1])
                    else: continue
                    
                    if any(d['content_id'] == content_id for d in scraped_data):
                        continue
                    
                    title_tag = article.select_one('h2')
                    title = title_tag.get_text(strip=True) if title_tag else ""
                    text_tag = article.select_one('p')
                    content_text = text_tag.get_text(strip=True) if text_tag else ""
                    date_tag = article.select_one('time')
                    date_str = date_tag.get_text(strip=True) if date_tag else ""
                    
                    scraped_data.append({
                        "content_id": content_id,
                        "platform": "everytime",
                        "keyword": keyword,
                        "title": title,
                        "content_text": content_text,
                        "posted_at_raw": date_str
                    })
                    match_count += 1
                except Exception:
                    continue
                    
            print(f"   -> [{page_depth}층] 파싱 완료: 새롭게 수집된 데이터 {match_count}개")
            
            # 3) [다음] 버튼 판별 및 클릭 시도
            try:
                # 제보해주신 셀렉터로 다음 버튼 요소를 찾습니다.
                # 단, 여러 페이징 버튼 중 '다음' 텍스트를 가진 요소를 정밀 매칭합니다.
                pagination_buttons = driver.find_elements(By.CSS_SELECTOR, "#container > div.wrap.articles > div.pagination > a")
                
                next_btn = None
                for btn in pagination_buttons:
                    if btn.text.strip() == "다음":
                        next_btn = btn
                        break
                
                # '다음' 버튼이 눈에 보이고 활성화되어 있다면 클릭
                if next_btn and next_btn.is_displayed():
                    print("   -> ➡️ [다음] 버튼 발견! 마우스 클릭으로 다음 그룹 페이지로 이동합니다.")
                    # 일반 click()이 씹힐 수 있으므로 자바스크립트로 확실하게 클릭을 주입합니다.
                    driver.execute_script("arguments[0].click();", next_btn)
                    page_depth += 1
                    time.sleep(4.0) # 다음 페이지 목록이 렌더링될 때까지 충분히 대기
                else:
                    print(f"   -> ✨ [끝] 더 이상 클릭할 '다음' 버튼이 없습니다. [{keyword}] 최종 종료.")
                    break
            except Exception:
                # 다음 버튼 탐색 실패 시 루프 종료 (최종 페이지 도달)
                print(f"   -> ✨ [끝] 페이징의 마지막 번호입니다. [{keyword}] 최종 종료.")
                break
                
        time.sleep(2.0)
                
    except Exception as e:
        print(f"❌ [{keyword}] 검색 제어 중 오류 발생: {e}")
        print("   -> 안전을 위해 메인 홈 주소로 리셋 기동합니다.")
        driver.get("https://everytime.kr/")
        time.sleep(4.0)
        continue

# -------------------------------------------------------------------------
# 3. 종합 데이터 CSV 저장 단계
# -------------------------------------------------------------------------
df = pd.DataFrame(scraped_data)

if not df.empty:
    output_filename = f"everytime_pagination_perfect_{datetime.now().strftime('%Y%m%d')}.csv"
    df.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"\n🎉 [대성공] 스크롤과 [다음] 버튼을 뚫고 진짜 전수 수집 완수! 총 {len(df)}건 저장 완료 -> '{output_filename}'")
else:
    print("\n⚠️ 수집된 데이터가 없습니다. 로그인 상태를 유지한 상태에서 다시 가동해 보세요.")

🔗 [연동 성공] 디버그 크롬 실시간 제어권 획득 완료!
🌐 안전한 메인 홈 화면으로 진입합니다.

🔎 [단기 임대] 검색 및 페이징 한정 해제 전수 수집 시작...
   -> 엔터 입력 완료. 검색 결과 첫 페이지 로딩 대기...
   -> 🛑 [1층] 내부 박스 끝까지 스크롤 다운 중...
   -> [1층] 파싱 완료: 새롭게 수집된 데이터 20개
   -> ➡️ [다음] 버튼 발견! 마우스 클릭으로 다음 그룹 페이지로 이동합니다.
   -> 🛑 [2층] 내부 박스 끝까지 스크롤 다운 중...
   -> [2층] 파싱 완료: 새롭게 수집된 데이터 20개
   -> ➡️ [다음] 버튼 발견! 마우스 클릭으로 다음 그룹 페이지로 이동합니다.
   -> 🛑 [3층] 내부 박스 끝까지 스크롤 다운 중...
   -> [3층] 파싱 완료: 새롭게 수집된 데이터 20개
   -> ➡️ [다음] 버튼 발견! 마우스 클릭으로 다음 그룹 페이지로 이동합니다.
   -> 🛑 [4층] 내부 박스 끝까지 스크롤 다운 중...
   -> [4층] 파싱 완료: 새롭게 수집된 데이터 20개
   -> ➡️ [다음] 버튼 발견! 마우스 클릭으로 다음 그룹 페이지로 이동합니다.
   -> 🛑 [5층] 내부 박스 끝까지 스크롤 다운 중...
   -> [5층] 파싱 완료: 새롭게 수집된 데이터 20개
   -> ➡️ [다음] 버튼 발견! 마우스 클릭으로 다음 그룹 페이지로 이동합니다.
   -> 🛑 [6층] 내부 박스 끝까지 스크롤 다운 중...
   -> [6층] 파싱 완료: 새롭게 수집된 데이터 20개
   -> ➡️ [다음] 버튼 발견! 마우스 클릭으로 다음 그룹 페이지로 이동합니다.
   -> 🛑 [7층] 내부 박스 끝까지 스크롤 다운 중...
   -> [7층] 파싱 완료: 새롭게 수집된 데이터 20개
   -> ➡️ [다음] 버튼 발견! 마우스 클릭으로 다음 그룹 페이지로 이동합니다.
   -> 🛑 [8층] 내부 박스 끝까지 스크롤 다운 중...
   -> [8층] 파싱 완료